In [ ]:

import os, sys, gc, glob, math, random, warnings, argparse
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import cv2
from PIL import Image
from tqdm import tqdm
from collections import OrderedDict
from scipy.special import expit
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim.lr_scheduler as sched
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import albumentations as A
import rasterio
from rasterio.enums import Resampling
import pydensecrf.densecrf as dcrf
from pydensecrf.utils import unary_from_softmax
from transformers import ViTModel
from transformers import Dinov2Model
import optuna
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    jaccard_score, cohen_kappa_score, roc_auc_score,
    average_precision_score, confusion_matrix, roc_curve,
    matthews_corrcoef, balanced_accuracy_score, precision_recall_curve
)
from sklearn.exceptions import UndefinedMetricWarning
warnings.filterwarnings("ignore", category=rasterio.errors.NotGeoreferencedWarning)
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

# ─── Logging ─────────────────────────────────────────────
import logging
os.makedirs("outputs", exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('outputs/training.log', encoding='utf-8'),
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger(__name__)

# ─── Arguments ───────────────────────────────────────────
parser = argparse.ArgumentParser()
parser.add_argument('--enable-deterministic', action='store_true')
parser.add_argument('--train_image1_path', default=os.getenv('TRAIN_IMG1_PATH', r"D:\Hi-CNA dataset\train\image1"))
parser.add_argument('--train_image2_path', default=os.getenv('TRAIN_IMG2_PATH', r"D:\Hi-CNA dataset\train\image2"))
parser.add_argument('--train_changes_path', default=os.getenv('TRAIN_CHANGES_PATH', r"D:\Hi-CNA dataset\train\change"))
parser.add_argument('--val_image1_path', default=os.getenv('VAL_IMG1_PATH', r"D:\Hi-CNA dataset\val\image1"))
parser.add_argument('--val_image2_path', default=os.getenv('VAL_IMG2_PATH', r"D:\Hi-CNA dataset\val\image2"))
parser.add_argument('--val_changes_path', default=os.getenv('VAL_CHANGES_PATH', r"D:\Hi-CNA dataset\val\change"))
parser.add_argument('--test_image1_path', default=os.getenv('TEST_IMG1_PATH', r"D:\Hi-CNA dataset\test\image1"))
parser.add_argument('--test_image2_path', default=os.getenv('TEST_IMG2_PATH', r"D:\Hi-CNA dataset\test\image2"))
parser.add_argument('--test_changes_path', default=os.getenv('TEST_CHANGES_PATH',r"D:\Hi-CNA dataset\test\change_255"))
parser.add_argument('--resume', default=None)
args, _ = parser.parse_known_args()

# ─── Config ──────────────────────────────────────────────
BATCH_SIZE = 4
ACCUM_STEPS = 4
NUM_EPOCHS = 30
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE = 224
PATCH_SIZE = 14                   
SEED = 42
NUM_WORKERS = 0
DROPOUT = 0.1
WEIGHT_DECAY = 0.05
NUM_CHANNELS = 5
MAX_VIS = 30
EMBED_DIM = 768
NUM_HEADS = 12
USE_CRF = True
CRF_INTERVAL = 10
NUM_PATCHES = (IMG_SIZE // PATCH_SIZE) ** 2   # now = 256
AMP_DTYPE = (torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16)
THRESHOLD = 0.5
assert IMG_SIZE % PATCH_SIZE == 0
logger.info(f"Device: {DEVICE} | AMP: {AMP_DTYPE} | Threshold: {THRESHOLD} (fixed) | DINOv2 patch=14 | NUM_PATCHES={NUM_PATCHES}")

# ─── Seed ────────────────────────────────────────────────
def set_seed(seed):
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    np.random.seed(seed); random.seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
set_seed(SEED)

# ─── GPU Utils ───────────────────────────────────────────
def clear_cache():
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    gc.collect()
def log_gpu_memory(tag=""):
    if DEVICE.type == "cuda":
        a = torch.cuda.memory_allocated() / 1e9
        r = torch.cuda.memory_reserved() / 1e9
        logger.info(f"GPU [{tag}] Alloc:{a:.2f}GB Res:{r:.2f}GB")

# ─── Data I/O ────────────────────────────────────────────
def read_image_rasterio(image_path):
    try:
        with rasterio.open(image_path) as src:
            if src.count != 4:
                raise ValueError(f"Expected 4 channels, got {src.count} in {image_path}")
            image = src.read(out_shape=(4, IMG_SIZE, IMG_SIZE), resampling=Resampling.bilinear)
            image = image.transpose(1, 2, 0).astype(np.float32)
            red = image[:, :, 0]
            nir = image[:, :, 3]
            ndvi = (nir - red) / (nir + red + 1e-8)
            ndvi = np.clip(ndvi, -1, 1)
            dtype = src.dtypes[0]
            max_val = 65535.0 if 'uint16' in dtype else 255.0 if 'uint8' in dtype else 1.0
            image = image / max_val
            image = np.nan_to_num(image, nan=0.0, posinf=1.0, neginf=0.0)
            image = np.clip(image, 0, 1)
            mean = image.mean(axis=(0, 1), keepdims=True)
            std = image.std(axis=(0, 1), keepdims=True) + 1e-8
            rgbnir_norm = (image - mean) / std
            rgbnir_norm = np.clip(rgbnir_norm, -3, 3)
            rgbnir_norm = (rgbnir_norm + 3) / 6
            ndvi_norm = (ndvi + 1) / 2.0
            ndvi_norm = ndvi_norm[..., np.newaxis]
            image_norm = np.dstack((rgbnir_norm, ndvi_norm))
            return image_norm
    except Exception as e:
        logger.error(f"Error reading {image_path}: {str(e)}")
        return np.zeros((IMG_SIZE, IMG_SIZE, 5), dtype=np.float32)

def read_mask_pil(path):
    try:
        m = Image.open(path).convert('L').resize((IMG_SIZE, IMG_SIZE), Image.Resampling.NEAREST)
        return (np.array(m) > 0).astype(np.float32)
    except Exception as e:
        logger.error(f"Error reading mask {path}: {e}")
        return np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float32)

def verify_pairs(img1, img2, masks=None):
    assert len(img1) == len(img2)
    if masks: assert len(img1) == len(masks)
    for i, (f1, f2) in enumerate(zip(img1, img2)):
        b1 = os.path.splitext(os.path.basename(f1))[0]
        b2 = os.path.splitext(os.path.basename(f2))[0]
        assert b1 == b2, f"Mismatch at {i}: {b1} vs {b2}"
    if masks:
        for i, (f1, fm) in enumerate(zip(img1, masks)):
            b1 = os.path.splitext(os.path.basename(f1))[0]
            bm = os.path.splitext(os.path.basename(fm))[0]
            assert b1 == bm, f"Mask mismatch at {i}: {b1} vs {bm}"
    logger.info(f" Pairs verified: {len(img1)} samples")

def load_files(path, ext):
    return sorted(glob.glob(os.path.join(path, f"*.{ext}")))

# ─── Augmentation ────────────────────────────────────────
class ConsistentColorJitter(A.ImageOnlyTransform):
    def __init__(self, brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05, p=0.5):
        super().__init__(p=p)
        self.brightness = brightness; self.contrast = contrast
        self.saturation = saturation; self.hue = hue
        self._j = A.ColorJitter(brightness=brightness, contrast=contrast,
                                saturation=saturation, hue=hue, p=1.0)
    def apply(self, img, seed=0, **params):
        random.seed(seed); np.random.seed(seed)
        rgb_aug = self._j(image=img[:, :, :3])['image']
        return np.concatenate([rgb_aug, img[:, :, 3:]], axis=2).astype(np.float32)
    def get_params(self): return {"seed": random.randint(0, 99999)}
    def get_transform_init_args_names(self): return ("brightness", "contrast", "saturation", "hue")

# ─── Dataset ─────────────────────────────────────────────
class ChangeDetectionDataset(Dataset):
    def __init__(self, img1_files, img2_files, mask_files=None,
                 transform=None, photo_transform=None, is_test=False, cache_size=None):
        self.img1_files = img1_files
        self.img2_files = img2_files
        self.mask_files = mask_files
        self.transform = transform
        self.photo_transform = photo_transform
        self.is_test = is_test
        self.cache = OrderedDict()
        self.cache_size = len(img1_files) // 2 if cache_size is None else cache_size
    def __len__(self): return len(self.img1_files)
    def _load(self, path, is_mask=False):
        if path in self.cache:
            self.cache.move_to_end(path); return self.cache[path].copy()
        data = read_mask_pil(path) if is_mask else read_image_rasterio(path)
        if self.cache_size > 0 and len(self.cache) >= self.cache_size:
            self.cache.popitem(last=False)
        self.cache[path] = data.copy()
        return data
    def _apply_photo(self, img):
        if self.photo_transform is None: return img
        rgb_aug = self.photo_transform(image=img[:, :, :3])['image']
        return np.concatenate([rgb_aug, img[:, :, 3:]], axis=2).astype(np.float32)
    def __getitem__(self, idx):
        img1 = self._load(self.img1_files[idx])
        img2 = self._load(self.img2_files[idx])
        has_mask = not (self.is_test or self.mask_files is None)
        if has_mask: mask = self._load(self.mask_files[idx], is_mask=True)
        if self.transform:
            if has_mask:
                t = self.transform(image=img1, image2=img2, mask=mask)
                img1, img2, mask = t['image'].copy(), t['image2'].copy(), t['mask'].copy()
            else:
                t = self.transform(image=img1, image2=img2)
                img1, img2 = t['image'].copy(), t['image2'].copy()
        img1 = self._apply_photo(img1)
        img2 = self._apply_photo(img2)
        result = {
            'image1': torch.from_numpy(img1.transpose(2, 0, 1)).float(),
            'image2': torch.from_numpy(img2.transpose(2, 0, 1)).float(),
            'filename': os.path.basename(self.img1_files[idx])
        }
        if has_mask: result['mask'] = torch.from_numpy(mask).float()
        return result

# ─── Custom Layers ───────────────────────────────────────
class DropPath(nn.Module):
    def __init__(self, drop_prob=0.1):
        super().__init__(); self.drop_prob = drop_prob
    def forward(self, x):
        if self.drop_prob == 0. or not self.training: return x
        keep = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        mask = keep + torch.rand(shape, dtype=x.dtype, device=x.device)
        return x.div(keep) * mask.floor_()

# ─── BalancedCrossAttention ───────────────────────────────
class BalancedCrossAttention(nn.Module):
    def __init__(self, dim, num_heads=NUM_HEADS, dropout=DROPOUT, num_patches=NUM_PATCHES):
        super().__init__()
        assert dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        for name in ['q1', 'k1', 'v1', 'q2', 'k2', 'v2']:
            setattr(self, f'{name}_proj', nn.Linear(dim, dim))
        self.diff_gate = nn.Sequential(
            nn.Linear(dim * 2, dim), nn.LayerNorm(dim), nn.GELU(),
            nn.Linear(dim, dim), nn.Sigmoid()
        )
        self.fusion = nn.Linear(dim * 2, dim)
        self.output_proj = nn.Linear(dim, dim)
        self.out_norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)
        gs = int(math.sqrt(num_patches))
        self.relative_bias_table = nn.Parameter(torch.zeros((2 * gs - 1) ** 2, num_heads))
        coords = torch.arange(gs)
        grid = torch.stack(torch.meshgrid(coords, coords, indexing='ij'), dim=-1).reshape(-1, 2)
        rel_coords = grid[:, None, :] - grid[None, :, :]
        rel_coords[..., 0] += gs - 1; rel_coords[..., 1] += gs - 1
        rel_coords[..., 0] *= 2 * (gs - 1) + 1
        self.register_buffer('rel_index', rel_coords.sum(-1).long())
        nn.init.trunc_normal_(self.relative_bias_table, std=0.02)
    def _split(self, x):
        B, N, _ = x.shape
        return x.view(B, N, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
    def _merge(self, x):
        return x.permute(0, 2, 1, 3).contiguous().view(x.shape[0], x.shape[2], -1)
    def forward(self, x1, x2):
        dg = self.diff_gate(torch.cat([x1, x2], dim=-1)) * 0.9 + 0.05
        x1m = x1 * dg + 0.1 * x1
        x2m = x2 * (1.0 - dg) + 0.1 * x2
        q1, k1, v1 = self._split(self.q1_proj(x1m)), self._split(self.k1_proj(x1m)), self._split(self.v1_proj(x1m))
        q2, k2, v2 = self._split(self.q2_proj(x2m)), self._split(self.k2_proj(x2m)), self._split(self.v2_proj(x2m))
        N = x1.shape[1]
        bias = self.relative_bias_table[self.rel_index.view(-1)].view(N, N, -1).permute(2, 0, 1).unsqueeze(0)
        a12 = F.softmax((q1 @ k2.transpose(-2, -1)) * self.scale + bias, dim=-1).type_as(x1)
        a21 = F.softmax((q2 @ k1.transpose(-2, -1)) * self.scale + bias, dim=-1).type_as(x1)
        sym = (a12 + a21.transpose(-2, -1)) / 2.0
        out_12 = self._merge(sym @ v2)
        out_21 = self._merge(sym @ v1)
        fused = self.fusion(torch.cat([out_12, out_21], dim=-1))
        return self.dropout(self.out_norm(self.output_proj(fused)))

# ─── TransformerBlock ────────────────────────────────────
class TransformerBlock(nn.Module):
    def __init__(self, dim=EMBED_DIM, num_heads=NUM_HEADS, mlp_ratio=4.0,
                 dropout=DROPOUT, drop_path=0.1, num_patches=NUM_PATCHES):
        super().__init__()
        self.norm1_x1 = nn.LayerNorm(dim)
        self.norm1_x2 = nn.LayerNorm(dim)
        self.attn = BalancedCrossAttention(dim, num_heads, dropout, num_patches)
        self.norm2 = nn.LayerNorm(dim)
        mlp_dim = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(mlp_dim, dim), nn.Dropout(dropout)
        )
        self.dp = DropPath(drop_path)
    def forward(self, x1, x2):
        attn_out = self.attn(self.norm1_x1(x1), self.norm1_x2(x2))
        x = x1 + self.dp(attn_out)
        x = x + self.dp(self.mlp(self.norm2(x)))
        return x

# ─── Encoder (DINOv2) ─────────────────────────────────────────────

class VisionTransformerEncoder(nn.Module):
    def __init__(self, in_chans=NUM_CHANNELS):
        super().__init__()
        self.vit = Dinov2Model.from_pretrained("facebook/dinov2-base")  
        old = self.vit.embeddings.patch_embeddings.projection
        new = nn.Conv2d(in_chans, old.out_channels, old.kernel_size, old.stride,
                        old.padding, bias=old.bias is not None)
        new.weight.data[:, :3] = old.weight.data
        avg = old.weight.data.mean(dim=1, keepdim=True)
        new.weight.data[:, 3:4] = avg.clone()
        new.weight.data[:, 4:5] = avg.clone()
        if old.bias is not None: new.bias.data = old.bias.data.clone()
        self.vit.embeddings.patch_embeddings.projection = new
        self.vit.config.num_channels = in_chans
        self.vit.embeddings.patch_embeddings.num_channels = in_chans
        for i in range(6):
            for p in self.vit.encoder.layer[i].parameters():
                p.requires_grad = False
        self.embed_dim = self.vit.config.hidden_size
        self.num_patches = (IMG_SIZE // PATCH_SIZE) ** 2

    def forward(self, x):
        
        out = self.vit(pixel_values=x, 
                       output_hidden_states=True, 
                       interpolate_pos_encoding=True)
        hs = out.hidden_states
        H = W = IMG_SIZE // PATCH_SIZE
        to_s = lambda h: h[:, 1:].transpose(1, 2).reshape(x.size(0), self.embed_dim, H, W)
        skips = [to_s(hs[i]) for i in [0, 4, 8, 12]]
        return hs[-1][:, 1:], skips

# ─── Siamese Model ───────────────────────────────────────

class SiameseViT(nn.Module):
    def __init__(self, in_chans=NUM_CHANNELS, embed_dim=EMBED_DIM,
                 num_heads=NUM_HEADS, mlp_ratio=4.0, dropout=DROPOUT):
        super().__init__()
        self.encoder = VisionTransformerEncoder(in_chans)
        np_ = self.encoder.num_patches
        self.cross_blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, dropout,
                             drop_path=0.1 * i / max(3, 1), num_patches=np_)
            for i in range(4)
        ])
        self.fusion = nn.Sequential(
            nn.Linear(embed_dim * 3, embed_dim), nn.GELU(),
            nn.LayerNorm(embed_dim), nn.Dropout(dropout),
            nn.Linear(embed_dim, embed_dim)
        )
        self.skip_proj0 = nn.Conv2d(embed_dim * 3, embed_dim * 2, 1)
        self.skip_proj1 = nn.Conv2d(embed_dim * 2 + embed_dim * 2, embed_dim, 1)
        self.skip_proj2 = nn.Conv2d(embed_dim + embed_dim * 2, embed_dim // 2, 1)
        self.skip_proj3 = nn.Conv2d(embed_dim // 2 + embed_dim * 2, embed_dim // 4, 1)
        def dec_block(ic, oc):
            return nn.Sequential(
                nn.Conv2d(ic, oc, 3, padding=1), nn.BatchNorm2d(oc), nn.ReLU(inplace=True),
                nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
            )
        self.decoder = nn.ModuleList([
            dec_block(embed_dim * 2, embed_dim * 2),
            dec_block(embed_dim, embed_dim),
            dec_block(embed_dim // 2, embed_dim // 2),
            dec_block(embed_dim // 4, embed_dim // 4),
            nn.Sequential(
                nn.Conv2d(embed_dim // 4, embed_dim // 8, 3, padding=1),
                nn.BatchNorm2d(embed_dim // 8), nn.ReLU(inplace=True),
                nn.Dropout2d(dropout), nn.Conv2d(embed_dim // 8, 1, 1)
            )
        ])

    def forward(self, img1, img2):
        B = img1.shape[0]
        x1, feat1 = self.encoder(img1)
        x2, feat2 = self.encoder(img2)
        for block in self.cross_blocks:
            x1_new = block(x1, x2)
            x2_new = block(x2, x1)
            x1, x2 = x1_new, x2_new
        diff = torch.abs(x1 - x2)
        x = self.fusion(torch.cat([x1, x2, diff], dim=-1))
        H = W = IMG_SIZE // PATCH_SIZE
        x = x.transpose(1, 2).reshape(B, EMBED_DIM, H, W)
        x = self.skip_proj0(torch.cat([x, torch.cat([feat1[0], feat2[0]], dim=1)], dim=1))
        x = self.decoder[0](x)
        for i in range(1, 4):
            skip = F.interpolate(torch.cat([feat1[i], feat2[i]], dim=1),
                                 size=x.shape[2:], mode='bilinear', align_corners=False)
            x = getattr(self, f'skip_proj{i}')(torch.cat([x, skip], dim=1))
            x = self.decoder[i](x)
        out = self.decoder[4](x)
        out = F.interpolate(out, size=(IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False)
        return out.squeeze(1)
# ─── Loss ────────────────────────────────────────────────
class FocalTverskyLoss(nn.Module):
    def __init__(self, alpha=0.7, beta=0.3, gamma=2.0, smooth=1e-5):
        super().__init__()
        self.alpha = alpha; self.beta = beta
        self.gamma = gamma; self.smooth = smooth
    def forward(self, inp, tgt):
        p = torch.clamp(torch.sigmoid(inp).view(-1), 1e-7, 1 - 1e-7)
        t = tgt.view(-1)
        TP = (p * t).sum()
        FP = ((1 - t) * p).sum()
        FN = (t * (1 - p)).sum()
        ti = (TP + self.smooth) / (TP + self.alpha * FP + self.beta * FN + self.smooth)
        return (1 - ti) ** self.gamma

class HybridLoss(nn.Module):
    def __init__(self, pos_weight):
        super().__init__()
        self.ft = FocalTverskyLoss()
        self.bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight).to(DEVICE))
    def update_pos_weight(self, pw):
        self.bce.pos_weight = torch.tensor(pw).to(DEVICE)
    def forward(self, inp, tgt):
        return 0.7 * self.ft(inp, tgt) + 0.3 * self.bce(inp, tgt)

# ─── Post-processing ─────────────────────────────────────
def post_process_prediction(pred, rgb_img=None, is_logits=True):
    if is_logits:
        probs = expit(pred)
    else:
        probs = np.array(pred)
    probs = np.clip(probs, 0, 1)
    pred_softmax = np.stack([1 - probs, probs], axis=0)
    unary = unary_from_softmax(pred_softmax)
    unary = np.ascontiguousarray(unary)
    d = dcrf.DenseCRF2D(IMG_SIZE, IMG_SIZE, 2)
    d.setUnaryEnergy(unary)
    d.addPairwiseGaussian(sxy=3, compat=5)
    if rgb_img is None:
        rgb_img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    rgb_img = np.ascontiguousarray(rgb_img)
    d.addPairwiseBilateral(sxy=80, srgb=13, rgbim=rgb_img, compat=12)
    Q = d.inference(3)
    pred_crf = np.argmax(Q, axis=0).reshape((IMG_SIZE, IMG_SIZE))
    return pred_crf.astype(np.uint8)

def morph_postprocess(pred, min_area=80, opening_ksize=3):
    pred = pred.astype(np.uint8)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (opening_ksize, opening_ksize))
    pred = cv2.morphologyEx(pred, cv2.MORPH_OPEN, k)
    n, lm, stats, _ = cv2.connectedComponentsWithStats(pred, connectivity=8)
    clean = np.zeros_like(pred)
    for lbl in range(1, n):
        if stats[lbl, cv2.CC_STAT_AREA] >= min_area:
            clean[lm == lbl] = 1
    return clean.astype(np.uint8)

# ─── Helpers ─────────────────────────────────────────────
def to_numpy(t): return t.detach().float().cpu().numpy()
def compute_pos_weight(mask_files, n=200):
    pos = neg = 0
    for f in random.sample(mask_files, min(n, len(mask_files))):
        m = read_mask_pil(f); pos += m.sum(); neg += (1 - m).sum()
    pw = float(neg / (pos + 1e-8))
    logger.info(f"pos_weight = {pw:.2f}")
    return pw
def build_scheduler(optimizer, warmup_epochs, total_epochs):
    return sched.SequentialLR(optimizer, [
        sched.LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=warmup_epochs),
        sched.CosineAnnealingLR(optimizer, T_max=total_epochs - warmup_epochs, eta_min=1e-6)
    ], milestones=[warmup_epochs])
def build_scaler():
    return GradScaler(device='cuda') if DEVICE.type == 'cuda' else GradScaler(device='cpu', enabled=False)
def make_loaders(bs, train_ds, val_ds, test_ds):
    kw = dict(num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=False, prefetch_factor=None)
    return (DataLoader(train_ds, batch_size=bs, shuffle=True, **kw),
            DataLoader(val_ds, batch_size=bs, shuffle=False, **kw),
            DataLoader(test_ds, batch_size=bs, shuffle=False, **kw))

# ─── Metrics ─────────────────────────────────────────────
def compute_metrics(labels, preds, probs, per_image_data=None):
    MAX_SAMPLES = 500_000
    if len(labels) > MAX_SAMPLES:
        idx = np.random.choice(len(labels), MAX_SAMPLES, replace=False)
        labels = labels[idx]
        preds = preds[idx]
        probs = probs[idx]
    probs = np.clip(probs.astype(np.float64), 0.0, 1.0)
    labels = labels.astype(np.int32)
    preds = preds.astype(np.int32)
    m = {
        'accuracy': accuracy_score(labels, preds),
        'balanced_accuracy': balanced_accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='binary', zero_division=0),
        'precision': precision_score(labels, preds, average='binary', zero_division=0),
        'recall': recall_score(labels, preds, average='binary', zero_division=0),
        'iou': jaccard_score(labels, preds, average='binary', zero_division=0),
        'kappa': cohen_kappa_score(labels, preds),
        'mcc': matthews_corrcoef(labels, preds),
    }
    if len(np.unique(labels)) >= 2:
        m['auc'] = roc_auc_score(labels, probs)
        m['ap'] = average_precision_score(labels, probs)
    else:
        m['auc'] = m['ap'] = 0.0
    if per_image_data:
        aucs = [roc_auc_score(l, p) for l, p in per_image_data if len(np.unique(l)) >= 2]
        m['mean_per_image_auc'] = float(np.mean(aucs)) if aucs else 0.0
    return m

# ─── Visualization ───────────────────────────────────────
def _vis_dir(base, epoch):
    d = os.path.join(base, f"epoch_{epoch:03d}", "visualizations")
    os.makedirs(d, exist_ok=True); return d
def _plots_dir(base, epoch):
    d = os.path.join(base, f"epoch_{epoch:03d}", "plots")
    os.makedirs(d, exist_ok=True); return d
def save_vis(img1, img2, mask, pred, fname, base, epoch):
    def stretch(img):
        lo, hi = np.percentile(img, (2, 98))
        return np.clip((img - lo) / (hi - lo + 1e-6), 0, 1)
    def clahe(u8):
        c = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8, 8))
        return cv2.merge([c.apply(ch) for ch in cv2.split(u8)])
    rgb1 = clahe((stretch(img1[:, :, :3]) * 255).astype(np.uint8))
    rgb2 = clahe((stretch(img2[:, :, :3]) * 255).astype(np.uint8))
    fig, ax = plt.subplots(1, 4, figsize=(16, 4))
    for a, im, t in zip(ax,
                        [rgb1, rgb2, (mask > 0.5).astype(np.uint8)*255, pred.astype(np.uint8)*255],
                        ["Image 1","Image 2","Ground Truth","Prediction"]):
        a.imshow(im, cmap='gray' if im.ndim == 2 else None); a.set_title(t); a.axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(_vis_dir(base, epoch), f"vis_{fname.replace('.tif','.png')}"),
                bbox_inches='tight', dpi=150)
    plt.close(fig)
def save_cm(labels, preds, phase, epoch, base):
    d = _plots_dir(base, epoch)
    cm = confusion_matrix(labels, preds)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
    else:
        tn = int(cm[0,0]) if cm.size else 0; fp = fn = tp = 0
    sep = "-" * 45
    print(f"\n{sep}\n CM — {phase} | Epoch {epoch+1}\n{sep}")
    print(f" {'':12s} Pred NO Pred YES")
    print(f" {'True NO':12s} {tn:>8d} {fp:>8d} (Spec:{tn/(tn+fp+1e-8):.4f})")
    print(f" {'True YES':12s} {fn:>8d} {tp:>8d} (Rec: {tp/(tp+fn+1e-8):.4f})")
    print(f" TP={tp:,} FP={fp:,} FN={fn:,} TN={tn:,}\n{sep}\n")
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["No Change","Change"], yticklabels=["No Change","Change"],
                linewidths=0.5, linecolor="gray", cbar_kws={"shrink": 0.8})
    ax.set_title(f"{phase} CM — Epoch {epoch+1}", fontsize=13, fontweight="bold", pad=12)
    ax.set_ylabel("True"); ax.set_xlabel("Predicted")
    for (r, c), val in np.ndenumerate(cm):
        lbl = 'TP' if r==1 and c==1 else ('TN' if r==0 and c==0 else ('FP' if r==0 else 'FN'))
        ax.text(c+0.5, r+0.7, f"({lbl})", ha="center", va="center", fontsize=9,
                color="white" if val > cm.max()*0.6 else "black", alpha=0.7)
    plt.tight_layout()
    path = os.path.join(d, f"{phase}_cm.png")
    plt.savefig(path, dpi=150, bbox_inches="tight"); plt.close(fig)
    logger.info(f"CM saved → {path}")
def save_roc(labels, probs, phase, epoch, base):
    d = _plots_dir(base, epoch)
    if len(np.unique(labels)) < 2: return
    try:
        fpr, tpr, thr = roc_curve(labels, probs)
        auc_val = roc_auc_score(labels, probs)
        ap_val = average_precision_score(labels, probs)
        bi = int(np.argmax(tpr - fpr))
        sep = "-" * 45
        print(f"{sep}\n ROC — {phase} | Epoch {epoch+1}\n{sep}")
        print(f" AUC:{auc_val:.4f} | AP:{ap_val:.4f}")
        print(f" Best Thr:{thr[bi]:.3f} → TPR={tpr[bi]:.4f} FPR={fpr[bi]:.4f}\n{sep}\n")
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
        ax1.plot(fpr, tpr, color="steelblue", lw=2, label=f"AUC={auc_val:.4f}")
        ax1.scatter(fpr[bi], tpr[bi], color="tomato", zorder=5, s=80, label=f"Thr={thr[bi]:.2f}")
        ax1.fill_between(fpr, tpr, alpha=0.08, color="steelblue")
        ax1.plot([0,1],[0,1],"k--",lw=1,alpha=0.5)
        ax1.set(title=f"{phase} ROC — E{epoch+1}", xlabel="FPR", ylabel="TPR",
                xlim=[-0.01,1.01], ylim=[-0.01,1.01])
        ax1.legend(); ax1.grid(alpha=0.3)
        prec, rec, _ = precision_recall_curve(labels, probs)
        ax2.plot(rec, prec, color="darkorange", lw=2, label=f"AP={ap_val:.4f}")
        ax2.fill_between(rec, prec, alpha=0.08, color="darkorange")
        ax2.axhline(labels.sum()/len(labels), color="k", linestyle="--", lw=1, alpha=0.5)
        ax2.set(title=f"{phase} PR — E{epoch+1}", xlabel="Recall", ylabel="Precision",
                xlim=[-0.01,1.01], ylim=[-0.01,1.01])
        ax2.legend(); ax2.grid(alpha=0.3)
        plt.tight_layout()
        path = os.path.join(d, f"{phase}_roc_pr.png")
        plt.savefig(path, dpi=150, bbox_inches="tight"); plt.close(fig)
        logger.info(f"ROC+PR saved → {path}")
    except Exception as e:
        logger.warning(f"save_roc failed ({phase} E{epoch+1}): {e}")
def fmt_metrics(phase, loss, m):
    return (f"{phase} - Loss:{loss:.4f} | Acc:{m.get('accuracy',0):.4f} | "
            f"F1:{m.get('f1',0):.4f} | Prec:{m.get('precision',0):.4f} | "
            f"Rec:{m.get('recall',0):.4f} | IoU:{m.get('iou',0):.4f} | "
            f"AUC:{m.get('auc',0):.4f} | AP:{m.get('ap',0):.4f}")
def print_epoch(epoch, t_loss, t_m, v_loss, v_m):
    sep = "=" * 75
    print(f"\n{sep}\n Epoch {epoch+1}/{NUM_EPOCHS}\n{sep}")
    print(fmt_metrics("Train", t_loss, t_m))
    print(fmt_metrics("Val ", v_loss, v_m) + f" | Thr:{THRESHOLD:.2f} (fixed)")
    print("-" * 75)
    for tag, m in [("Train", t_m), ("Val ", v_m)]:
        line = (f" {tag} → Kappa:{m.get('kappa',0):.4f} | MCC:{m.get('mcc',0):.4f} | "
                f"BalAcc:{m.get('balanced_accuracy',0):.4f}")
        if 'mean_per_image_auc' in m: line += f" | MeanImgAUC:{m['mean_per_image_auc']:.4f}"
        print(line)
    print(sep)

# ─── Training ────────────────────────────────────────────
def train_epoch(model, loader, optimizer, criterion, scaler, device, epoch, initial_pw):
    model.train()
    total_loss = 0.0
    sample_probs = []
    sample_labels = []
    optimizer.zero_grad()
    criterion.update_pos_weight(initial_pw * max(0.5, 1.0 - epoch * 0.01))
    for i, batch in enumerate(tqdm(loader, desc=f"Train E{epoch+1}")):
        img1 = batch['image1'].to(device, non_blocking=True)
        img2 = batch['image2'].to(device, non_blocking=True)
        mask = batch['mask'].to(device, non_blocking=True)
        with autocast(device_type=device.type, dtype=AMP_DTYPE):
            out = model(img1, img2)
            loss = criterion(out, mask) / ACCUM_STEPS
        loss = torch.nan_to_num(loss, nan=0., posinf=1e3, neginf=-1e3)
        if torch.isnan(loss) or torch.isinf(loss):
            logger.warning(f"NaN/Inf at batch {i}. Skipping.")
            optimizer.zero_grad(); continue
        scaler.scale(loss).backward()
        if (i + 1) % ACCUM_STEPS == 0 or (i + 1) == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
        total_loss += loss.item() * ACCUM_STEPS
        if i % 10 == 0:
            with torch.no_grad():
                probs_b = torch.sigmoid(out).float().cpu().numpy()
                mask_b = mask.float().cpu().numpy()
                sample_probs.append(probs_b.reshape(probs_b.shape[0], -1).mean(axis=1))
                sample_labels.append((mask_b.reshape(mask_b.shape[0], -1).mean(axis=1) > 0.1).astype(np.int32))
    if sample_probs:
        agg_probs = np.concatenate(sample_probs).astype(np.float64)
        agg_labels = np.concatenate(sample_labels).astype(np.int32)
        agg_preds = (agg_probs > THRESHOLD).astype(np.int32)
        metrics = compute_metrics(agg_labels, agg_preds, agg_probs)
    else:
        agg_probs = np.array([0.5])
        agg_labels = np.array([0])
        agg_preds = np.array([0])
        metrics = {k: 0.0 for k in ['accuracy','balanced_accuracy','f1','precision',
                                    'recall','iou','kappa','mcc','auc','ap']}
    metrics['best_threshold'] = THRESHOLD
    avg_loss = total_loss / max(len(loader), 1)
    return avg_loss, metrics, agg_labels, agg_preds, agg_probs

def evaluate(model, loader, criterion, device, epoch,
             phase="Val", base_dir="outputs", max_vis=MAX_VIS):
    model.eval()
    total_loss = 0.0
    logits_list, labels_list, per_image_data, vis_data = [], [], [], []
    vis_count = 0
    with torch.no_grad():
        for batch in tqdm(loader, desc=phase):
            img1 = batch['image1'].to(device, non_blocking=True)
            img2 = batch['image2'].to(device, non_blocking=True)
            mask = batch['mask'].to(device, non_blocking=True)
            with autocast(device_type=device.type, dtype=AMP_DTYPE):
                out = model(img1, img2)
                loss = criterion(out, mask)
            total_loss += torch.nan_to_num(loss, nan=0., posinf=1e3, neginf=-1e3).item()
            out_np = to_numpy(out); mask_np = to_numpy(mask)
            img1_np = to_numpy(batch['image1']).transpose(0, 2, 3, 1)
            img2_np = to_numpy(batch['image2']).transpose(0, 2, 3, 1)
            logits_list.append(out_np.reshape(out_np.shape[0], -1))
            labels_list.append(mask_np.reshape(mask_np.shape[0], -1))
            for i, fname in enumerate(batch['filename']):
                lbl_i = mask_np[i].flatten().astype(np.int32)
                if len(np.unique(lbl_i)) >= 2:
                    per_image_data.append((lbl_i, expit(out_np[i]).flatten()))
                if vis_count < max_vis:
                    vis_data.append((img1_np[i], img2_np[i], mask_np[i], out_np[i].copy(), fname))
                    vis_count += 1
    all_logits = np.concatenate(logits_list, axis=0).flatten()
    all_labels = np.concatenate(labels_list, axis=0).flatten().astype(np.int32)
    all_probs = expit(all_logits).astype(np.float64)
    all_preds = (all_probs > THRESHOLD).astype(np.int32)
    metrics = compute_metrics(all_labels, all_preds, all_probs, per_image_data)
    metrics['best_threshold'] = THRESHOLD
    for (img1_i, img2_i, mask_i, logits_i, fname) in vis_data:
        if epoch >= 12 and random.random() < 0.1:
            pred_vis = post_process_prediction(logits_i, rgb_img=(img1_i[:,:,:3]*255).astype(np.uint8), is_logits=True)
        else:
            pred_vis = (expit(logits_i) > THRESHOLD).astype(np.uint8)
        save_vis(img1_i, img2_i, mask_i, morph_postprocess(pred_vis), fname, base_dir, epoch)
    return total_loss / max(len(loader), 1), metrics, all_labels, all_preds, all_probs

def run_test(model, loader, device, epoch, base_dir="outputs"):
    model.eval()
    preds_list, labels_list, probs_list, per_image_data = [], [], [], []
    has_mask = False; img_count = 0
    pred_dir = os.path.join(base_dir, "predicted_maps")
    os.makedirs(pred_dir, exist_ok=True)
    with torch.no_grad():
        for batch in tqdm(loader, desc="Test"):
            img1 = batch['image1'].to(device, non_blocking=True)
            img2 = batch['image2'].to(device, non_blocking=True)
            has_mask = 'mask' in batch
            with autocast(device_type=device.type, dtype=AMP_DTYPE):
                out = model(img1, img2)
            out_np = to_numpy(out); probs_np = expit(out_np)
            img1_np = to_numpy(batch['image1']).transpose(0, 2, 3, 1)
            img2_np = to_numpy(batch['image2']).transpose(0, 2, 3, 1)
            mask_np = to_numpy(batch['mask']) if has_mask else np.zeros_like(probs_np)
            for i, fname in enumerate(batch['filename']):
                if USE_CRF and img_count % CRF_INTERVAL == 0:
                    pred_i = post_process_prediction(out_np[i], rgb_img=(img1_np[i,:,:,:3]*255).astype(np.uint8), is_logits=True)
                else:
                    pred_i = (probs_np[i] > THRESHOLD).astype(np.uint8)
                pred_i = morph_postprocess(pred_i)
                cv2.imwrite(os.path.join(pred_dir, f"pred_{fname.replace('.tif','.png')}"),
                            (pred_i*255).astype(np.uint8))
                save_vis(img1_np[i], img2_np[i], mask_np[i], pred_i, fname,
                         base_dir, epoch)
                if has_mask:
                    lbl_i = mask_np[i].flatten().astype(np.int32)
                    preds_list.append(pred_i.flatten())
                    labels_list.append(lbl_i)
                    probs_list.append(probs_np[i].flatten())
                    if len(np.unique(lbl_i)) >= 2:
                        per_image_data.append((lbl_i, probs_np[i].flatten()))
                img_count += 1
    if has_mask and preds_list:
        all_labels = np.concatenate(labels_list).astype(np.int32)
        all_probs = np.concatenate(probs_list).astype(np.float64)
        final_preds = (all_probs > THRESHOLD).astype(np.int32)
        metrics = compute_metrics(all_labels, final_preds, all_probs, per_image_data)
        metrics['best_threshold'] = THRESHOLD
        logger.info(f"Test → F1:{metrics['f1']:.4f} IoU:{metrics['iou']:.4f} AUC:{metrics['auc']:.4f}")
        save_cm(all_labels, final_preds, "Test", epoch, base_dir)
        save_roc(all_labels, all_probs, "Test", epoch, base_dir)
        return metrics, all_labels, final_preds, all_probs
    logger.info("No ground truth for test set.")
    return None, None, None, None

# ─── Checkpoint ──────────────────────────────────────────
def save_checkpoint(model, optimizer, scheduler, epoch, metrics, is_best=False, out_dir="outputs/final"):
    os.makedirs(out_dir, exist_ok=True)
    ckpt = {
        'epoch': epoch, 'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
        'metrics': metrics, 'best_f1': metrics.get('f1', 0.0),
        'threshold': THRESHOLD,
        'rng': {'torch': torch.get_rng_state(),
                'cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
                'numpy': np.random.get_state(), 'python': random.getstate()}
    }
    torch.save(ckpt, os.path.join(out_dir, f'ckpt_epoch_{epoch:03d}.pth'))
    if is_best:
        torch.save(ckpt, os.path.join(out_dir, 'best_model.pth'))
        logger.info(f"✅ Best model saved — F1:{metrics['f1']:.4f} | Thr:{THRESHOLD:.2f} (fixed)")
def load_checkpoint(model, optimizer, scheduler, path, device):
    if not os.path.exists(path):
        logger.warning(f"Checkpoint not found: {path}"); return 0, 0.0
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    if scheduler and ckpt.get('scheduler_state_dict'):
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    if 'rng' in ckpt:
        rs = ckpt['rng']
        torch.set_rng_state(rs['torch'])
        if rs.get('cuda') and torch.cuda.is_available(): torch.cuda.set_rng_state_all(rs['cuda'])
        np.random.set_state(rs['numpy']); random.setstate(rs['python'])
    start = ckpt.get('epoch', 0) + 1
    logger.info(f"Resumed from epoch {start} | F1={ckpt.get('best_f1',0):.4f} | Thr:{THRESHOLD:.2f} (fixed)")
    return start, ckpt.get('best_f1', 0.0)

# ─── Monitor ─────────────────────────────────────────────
class TrainingMonitor:
    def __init__(self, patience=30, min_delta=1e-4):
        self.patience = patience; self.min_delta = min_delta
        self.best_f1 = 0.0; self.no_improve = 0
        self.train_hist = {'loss': [], 'metrics': []}
        self.val_hist = {'loss': [], 'metrics': []}
    def update(self, tl, vl, tm, vm, epoch):
        self.train_hist['loss'].append(tl); self.train_hist['metrics'].append(tm)
        self.val_hist['loss'].append(vl); self.val_hist['metrics'].append(vm)
        improved = vm['f1'] > self.best_f1 + self.min_delta
        if improved: self.best_f1 = vm['f1']; self.no_improve = 0
        else: self.no_improve += 1
        if abs(tl - vl) > 0.15 or abs(tm['f1'] - vm['f1']) > 0.25:
            logger.warning(f"Overfitting E{epoch+1} | loss_gap={abs(tl-vl):.3f} f1_gap={abs(tm['f1']-vm['f1']):.3f}")
        return improved, self.no_improve >= self.patience
    def plot(self, path='outputs/curves.png'):
        fig, ax = plt.subplots(2, 3, figsize=(15, 10))
        ax[0,0].plot(self.train_hist['loss'], label='Train')
        ax[0,0].plot(self.val_hist['loss'], label='Val')
        ax[0,0].set_title('Loss'); ax[0,0].legend()
        for i, k in enumerate(['f1','iou','accuracy','precision','recall']):
            r, c = (i+1)//3, (i+1)%3
            ax[r,c].plot([m.get(k,0) for m in self.train_hist['metrics']], label='Train')
            ax[r,c].plot([m.get(k,0) for m in self.val_hist['metrics']], label='Val')
            ax[r,c].set_title(k.title()); ax[r,c].legend()
        plt.tight_layout(); plt.savefig(path, dpi=150, bbox_inches='tight'); plt.close()

# ─── Summary ─────────────────────────────────────────────
def save_summary(metrics, out_dir):
    if not metrics: return
    os.makedirs(out_dir, exist_ok=True)
    pd.DataFrame([{k: v for k, v in metrics.items() if isinstance(v, float)}]).to_csv(
        os.path.join(out_dir, "test_metrics.csv"), index=False, float_format='%.4f')
    with open(os.path.join(out_dir, "test_summary.txt"), "w", encoding="utf-8") as f:
        f.write("=== FINAL TEST RESULTS ===\n\n")
        f.write(f"Threshold: {THRESHOLD:.2f} (fixed)\n\n")
        for k, v in metrics.items():
            if isinstance(v, float): f.write(f"{k:25s}: {v:.4f}\n")
    print("\n" + "="*75 + "\nFINAL TEST RESULTS\n" + "="*75)
    print(f" {'threshold':25s}: {THRESHOLD:.4f} (fixed)")
    for k in ['accuracy','balanced_accuracy','f1','precision','recall','iou','kappa','mcc','auc','ap','mean_per_image_auc']:
        if k in metrics: print(f" {k:25s}: {metrics[k]:.4f}")
    print("="*75)

# ─── Optuna ──────────────────────────────────────────────
def run_optuna(train_ds, val_ds, test_ds, initial_pos_weight):
    def objective(trial):
        lr = trial.suggest_float('lr', 1e-6, 1e-3, log=True)
        wd = trial.suggest_float('weight_decay', 1e-6, 0.1, log=True)
        dropout = trial.suggest_float('dropout', 0.0, 0.3)
        batch_size = trial.suggest_categorical('batch_size', [4, 8, 16])
        n_epochs = 30; warmup_ep = 3
        t_ldr, v_ldr, _ = make_loaders(batch_size, train_ds, val_ds, test_ds)
        model = SiameseViT(dropout=dropout).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd, betas=(0.9,0.999), eps=1e-8)
        criterion = HybridLoss(pos_weight=initial_pos_weight)
        scaler = build_scaler()
        scheduler = build_scheduler(optimizer, warmup_ep, n_epochs)
        monitor = TrainingMonitor(patience=10)
        best_f1 = 0.0
        best_val_probs: np.ndarray | None = None
        best_val_labels: np.ndarray | None = None
        trial_dir = os.path.join("outputs", f"trial_{trial.number:03d}")
        os.makedirs(trial_dir, exist_ok=True)
        for epoch in range(n_epochs):
            clear_cache(); log_gpu_memory(f"T{trial.number} E{epoch+1}")
            t_loss, t_m, _, _, _ = train_epoch(model, t_ldr, optimizer, criterion, scaler, DEVICE, epoch, initial_pos_weight)
            v_loss, v_m, v_lbl, _, v_probs = evaluate(model, v_ldr, criterion, DEVICE, epoch, "Val", trial_dir)
            print_epoch(epoch, t_loss, t_m, v_loss, v_m)
            scheduler.step()
            improved, stop = monitor.update(t_loss, v_loss, t_m, v_m, epoch)
            if improved:
                best_f1 = float(v_m['f1'])
                best_val_probs = v_probs.copy(); best_val_labels = v_lbl.copy()
                logger.info(f"[T{trial.number}] New best F1:{best_f1:.4f} | Thr:{THRESHOLD:.2f} (fixed)")
            if stop: break
        trial.set_user_attr("val_f1", best_f1)
        if best_val_probs is not None:
            bm = compute_metrics(best_val_labels, (best_val_probs > THRESHOLD).astype(np.int32), best_val_probs)
            trial.set_user_attr("val_iou", float(bm.get('iou', 0.0)))
            trial.set_user_attr("val_auc", float(bm.get('auc', 0.0)))
        ckpt_path = os.path.join(trial_dir, "best_trial_model.pth")
        torch.save({
            'model_state_dict': model.state_dict(),
            'best_f1': best_f1,
            'threshold': THRESHOLD,
            'dropout': dropout,
            'batch_size': batch_size,
        }, ckpt_path)
        trial.set_user_attr("ckpt_path", ckpt_path)
        logger.info(f"[T{trial.number}] DONE Val F1:{best_f1:.4f} | Thr:{THRESHOLD:.2f} (fixed)")
        del model, optimizer, criterion, scaler, scheduler
        best_val_probs = best_val_labels = None
        clear_cache(); log_gpu_memory(f"After T{trial.number}")
        return best_f1
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=20)
    df = study.trials_dataframe(attrs=('number','value','params','user_attrs','state'))
    df.to_excel(os.path.join("outputs","final","optuna_trials.xlsx"), index=False)
    best = study.best_trial
    logger.info(
        f"\n{'='*65}\nOptuna done — {len(study.trials)} trials\n{'='*65}\n"
        f" Best Trial : #{best.number} Val F1:{best.value:.4f} Thr:{THRESHOLD:.2f} (fixed)\n"
        f" lr:{best.params['lr']:.2e} wd:{best.params['weight_decay']:.2e} "
        f"dropout:{best.params['dropout']:.3f} bs:{best.params['batch_size']}\n{'='*65}"
    )
    best_ckpt = best.user_attrs.get('ckpt_path', None)
    if best_ckpt and os.path.exists(best_ckpt):
        logger.info(f"Running Test on best trial #{best.number}...")
        ckpt_data = torch.load(best_ckpt, map_location=DEVICE, weights_only=False)
        best_model = SiameseViT(dropout=ckpt_data['dropout']).to(DEVICE)
        best_model.load_state_dict(ckpt_data['model_state_dict'])
        _, _, ts_ldr = make_loaders(ckpt_data['batch_size'], train_ds, val_ds, test_ds)
        test_out_dir = os.path.join("outputs", f"trial_{best.number:03d}", "test")
        ts_m, _, _, _ = run_test(best_model, ts_ldr, DEVICE, 0, base_dir=test_out_dir)
        if ts_m:
            save_summary(ts_m, test_out_dir)
            logger.info(f"Best Trial Test → F1:{ts_m['f1']:.4f} IoU:{ts_m['iou']:.4f} AUC:{ts_m['auc']:.4f}")
        del best_model; clear_cache()
    else:
        logger.warning("Best trial checkpoint not found.")
    return {**study.best_params}

# ─── Main ────────────────────────────────────────────────
def main():
    logger.info("Loading dataset paths...")
    for p in [args.train_image1_path, args.train_image2_path, args.train_changes_path,
              args.val_image1_path, args.val_image2_path, args.val_changes_path,
              args.test_image1_path, args.test_image2_path]:
        if p and not os.path.exists(p):
            raise FileNotFoundError(f"Path not found: {p}")
    train_img1 = load_files(args.train_image1_path, 'tif')
    train_img2 = load_files(args.train_image2_path, 'tif')
    train_masks = load_files(args.train_changes_path, 'png')
    val_img1 = load_files(args.val_image1_path, 'tif')
    val_img2 = load_files(args.val_image2_path, 'tif')
    val_masks = load_files(args.val_changes_path, 'png')
    test_img1 = load_files(args.test_image1_path, 'tif')
    test_img2 = load_files(args.test_image2_path, 'tif')
    test_masks = (load_files(args.test_changes_path, 'png')
                   if args.test_changes_path and os.path.exists(args.test_changes_path) else None)
    verify_pairs(train_img1, train_img2, train_masks)
    verify_pairs(val_img1, val_img2, val_masks)
    verify_pairs(test_img1, test_img2, test_masks)
    initial_pos_weight = compute_pos_weight(train_masks)
    geo_tfm = A.Compose([
        A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
        A.Affine(scale=(0.85,1.15), translate_percent={"x":(-0.05,0.05),"y":(-0.05,0.05)},
                 rotate=(-30,30), shear=(-5,5), p=0.5),
        A.ElasticTransform(alpha=60, sigma=6, p=0.2),
        A.GridDistortion(num_steps=5, distort_limit=0.2, p=0.15),
        A.OpticalDistortion(distort_limit=0.2, p=0.2),
        A.CoarseDropout(num_holes_range=(1,6), hole_height_range=(16,32),
                        hole_width_range=(16,32), fill=0, p=0.2),
    ], additional_targets={'image2':'image','mask':'mask'})
    photo_tfm = A.Compose([
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10, p=0.25),
        A.RandomGamma(gamma_limit=(80,120), p=0.3),
        A.RandomToneCurve(scale=0.1, p=0.2),
        A.OneOf([A.GaussianBlur(blur_limit=(3,7),p=1.), A.MotionBlur(blur_limit=7,p=1.), A.MedianBlur(blur_limit=5,p=1.)], p=0.3),
        A.OneOf([A.GaussNoise(p=1.), A.MultiplicativeNoise(multiplier=(0.9,1.1),p=1.)], p=0.3),
        A.OneOf([A.RandomFog(fog_coef_range=(0.1,0.3),p=1.), A.RandomShadow(shadow_roi=(0,0,1,1),shadow_dimension=5,p=1.)], p=0.15),
    ])
    train_ds = ChangeDetectionDataset(train_img1, train_img2, train_masks, transform=geo_tfm, photo_transform=photo_tfm)
    val_ds = ChangeDetectionDataset(val_img1, val_img2, val_masks)
    test_ds = ChangeDetectionDataset(test_img1, test_img2, test_masks, is_test=(test_masks is None))
    best_params = run_optuna(train_ds, val_ds, test_ds, initial_pos_weight)
    best_lr = best_params['lr']
    best_wd = best_params['weight_decay']
    best_dr = best_params['dropout']
    best_bs = best_params['batch_size']
    logger.info(f"Best params: {best_params} | Threshold: {THRESHOLD:.2f} (fixed)")
    t_ldr, v_ldr, ts_ldr = make_loaders(best_bs, train_ds, val_ds, test_ds)
    model = SiameseViT(dropout=best_dr).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=best_lr, weight_decay=best_wd, betas=(0.9,0.999), eps=1e-8)
    criterion = HybridLoss(pos_weight=initial_pos_weight)
    scaler = build_scaler()
    scheduler = build_scheduler(optimizer, 5, NUM_EPOCHS)
    monitor = TrainingMonitor(patience=30)
    final_dir = os.path.join("outputs", "final")
    os.makedirs(final_dir, exist_ok=True)
    start_epoch = 0; best_f1 = 0.0; last_epoch = 0
    if args.resume:
        start_epoch, best_f1 = load_checkpoint(model, optimizer, scheduler, args.resume, DEVICE)
    for epoch in range(start_epoch, NUM_EPOCHS):
        last_epoch = epoch
        clear_cache(); log_gpu_memory(f"Epoch {epoch+1}")
        t_loss, t_m, t_lbl, t_pred, t_prob = train_epoch(model, t_ldr, optimizer, criterion, scaler, DEVICE, epoch, initial_pos_weight)
        v_loss, v_m, v_lbl, v_pred, v_prob = evaluate(model, v_ldr, criterion, DEVICE, epoch, "Val", final_dir)
        print_epoch(epoch, t_loss, t_m, v_loss, v_m)
        save_cm(t_lbl, t_pred, "Train", epoch, final_dir); save_roc(t_lbl, t_prob, "Train", epoch, final_dir)
        save_cm(v_lbl, v_pred, "Val", epoch, final_dir); save_roc(v_lbl, v_prob, "Val", epoch, final_dir)
        scheduler.step()
        improved, stop = monitor.update(t_loss, v_loss, t_m, v_m, epoch)
        if improved:
            best_f1 = v_m['f1']
            logger.info(f"New best F1:{best_f1:.4f} | Thr:{THRESHOLD:.2f} (fixed)")
        save_checkpoint(model, optimizer, scheduler, epoch, v_m, is_best=improved, out_dir=final_dir)
        if epoch == start_epoch + 10:
            for p in model.encoder.vit.parameters(): p.requires_grad = True
            vit_p = [p for p in model.encoder.vit.parameters()]
            vit_ids = {id(p) for p in vit_p}
            rest_p = [p for p in model.parameters() if id(p) not in vit_ids]
            optimizer = torch.optim.AdamW([
                {'params': rest_p, 'lr': best_lr},
                {'params': vit_p, 'lr': 2e-5}
            ], weight_decay=best_wd, betas=(0.9,0.999), eps=1e-8)
            remaining = NUM_EPOCHS - epoch - 1
            if remaining > 0:
                scheduler = sched.CosineAnnealingLR(optimizer, T_max=remaining, eta_min=1e-6)
            logger.info(f"ViT unfrozen | rest LR={best_lr:.2e} ViT LR=2e-5 | Cosine {remaining} epochs")
        if stop: logger.info(f"Early stopping at epoch {epoch+1}"); break
    monitor.plot(os.path.join(final_dir, "training_curves.png"))
    logger.info(f"Final test — Threshold: {THRESHOLD:.2f} (fixed)")
    ts_m, _, _, _ = run_test(model, ts_ldr, DEVICE, last_epoch, base_dir=final_dir)
    if ts_m: save_summary(ts_m, final_dir)
    logger.info("Training complete ✅")

# ─── Entry Point ─────────────────────────────────────────
try:
    main()
except Exception as e:
    logger.error(f"Fatal error: {e}", exc_info=True)
    raise

2026-03-19 05:03:01,196 - INFO - Device: cuda | AMP: torch.bfloat16 | Threshold: 0.5 (fixed) | DINOv2 patch=14 | NUM_PATCHES=256
2026-03-19 05:03:01,206 - INFO - Loading dataset paths...
2026-03-19 05:03:01,294 - INFO - ✅ Pairs verified: 4080 samples
2026-03-19 05:03:01,318 - INFO - ✅ Pairs verified: 1358 samples
2026-03-19 05:03:01,332 - INFO - ✅ Pairs verified: 1358 samples
2026-03-19 05:03:01,522 - INFO - pos_weight = 4.84


[I 2026-03-19 05:03:01,531] A new study created in memory with name: no-name-66c9fb30-bf34-40bf-9016-0881bd8d047c


2026-03-19 05:03:01,959 - INFO - HTTP Request: HEAD https://huggingface.co/facebook/dinov2-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-19 05:03:02,016 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/dinov2-base/f9e44c814b77203eaa57a6bdbbd535f21ede1415/config.json "HTTP/1.1 200 OK"
2026-03-19 05:03:02,211 - INFO - HTTP Request: HEAD https://huggingface.co/facebook/dinov2-base/resolve/main/model.safetensors "HTTP/1.1 302 Found"


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

2026-03-19 05:03:03,142 - INFO - GPU [T0 E1] Alloc:0.69GB Res:0.76GB


Val: 100%|██████████| 170/170 [01:22<00:00,  2.06it/s]


... (201 outputs left out)

Val: 100%|██████████| 85/85 [00:43<00:00,  1.95it/s]



 Epoch 6/30
Train - Loss:0.3792 | Acc:0.8269 | F1:0.6814 | Prec:0.9506 | Rec:0.5310 | IoU:0.5168 | AUC:0.9562 | AP:0.9177
Val  - Loss:0.4280 | Acc:0.9097 | F1:0.7430 | Prec:0.6351 | Rec:0.8950 | IoU:0.5911 | AUC:0.9671 | AP:0.8637 | Thr:0.50 (fixed)
---------------------------------------------------------------------------
 Train → Kappa:0.5753 | MCC:0.6213 | BalAcc:0.7581
 Val  → Kappa:0.6901 | MCC:0.7050 | BalAcc:0.9036 | MeanImgAUC:0.9223
2026-03-19 15:42:59,593 - INFO - GPU [T3 E7] Alloc:1.84GB Res:2.18GB


Train E7:  73%|███████▎  | 185/255 [03:41<01:23,  1.20s/it]
[W 2026-03-19 15:46:41,361] Trial 3 failed with parameters: {'lr': 4.0872251220228056e-05, 'weight_decay': 0.00032734357180787006, 'dropout': 0.045130071394189716, 'batch_size': 16} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "C:\Users\LAPTOP\anaconda3\envs\Deep\lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\LAPTOP\AppData\Local\Temp\ipykernel_15996\1732439723.py", line 850, in objective
    t_loss, t_m, _, _, _ = train_epoch(model, t_ldr, optimizer, criterion, scaler, DEVICE, epoch, initial_pos_weight)
  File "C:\Users\LAPTOP\AppData\Local\Temp\ipykernel_15996\1732439723.py", line 622, in train_epoch
    for i, batch in enumerate(tqdm(loader, desc=f"Train E{epoch+1}")):
  File "C:\Users\LAPTOP\anaconda3\envs\Deep\lib\site-packages\tqdm\std.py", line 1181, in __iter__
    for obj in iterable:
  File "C:\

KeyboardInterrupt: 